In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
import pandas as pd

data_path = "../../experiment_data/balance_metrics/tvcg/multiseed/loss_1e6.csv"

data = pd.read_csv(data_path)
data = data[((data["model"] == "resnet_a") & (data["epoch"] == 128)) | ((data["model"] == "resnet_b") & (data["epoch"] == 144)) | ((data["model"] == "resnet_c") & (data["epoch"] == 136))]

with open('../../experiments/layer_orders_cross_layer.yml', 'r') as f:
    layer_order = yaml.safe_load(f)
    
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

In [4]:
data["layer"].unique()

array(['a3', 'a5', 'a4', 'a2', 'a1'], dtype=object)

In [5]:
resnet_names = ["layer1", "layer2", "layer3", "layer4", "classifier"]

In [18]:
resnet = data[(data["model"].str.contains("resnet")) & (data["split"] == "trainUval")].copy()

resnet["layer_idx"] = resnet["layer"].apply(lambda x: layer_order['resnet'].index(x))
resnet["name"] = resnet["model"].apply(lambda x: "Run " + x.split("_")[-1].capitalize())
resnet.sort_values(["layer_idx", "name"], inplace=True)

fig = px.line(resnet, x="layer", y="sackin_index", color="name", color_discrete_sequence=px.colors.qualitative.Dark24)
fig.update_traces(marker={'opacity': 0})
fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=300*2, height=180*2, font=dict(size=22), showlegend=True, legend=dict(
    xanchor="right", yanchor="top", x=0.98, y=0.98, bgcolor="rgba(255,255,255,0.5)", font=dict(size=15), title=""))
fig.update_xaxes(title_text="", automargin=True, ticktext=resnet_names, tickvals=resnet["layer"].unique(), title_standoff=0, tickangle=75)
fig.update_yaxes(title_text="Sackin Index", title_standoff=18, automargin=True)
fig.write_image(f"multiseed-blocks.png", scale=4)
fig.show()